# Модуль Б

Перед кодированием прочитайте «Задачи полигона» (задача 4), «Задачи кибериммунности» (CybP_01), «Python API участника» и «G-Code: механика и команды». Сначала завершите разгрузку корабля и доставьте основной груз, затем восстановите повреждённые кадры камеры и зарегистрируйте контейнер по кадру, для которого обнаружена ошибка.

Подготовьте прогон в «Симуляторе», выполните ячейки сверху вниз и нажмите «Запустить». Для зачётного прогона не задавайте seed и не используйте паузу. Описание алгоритма и причину выбранных проверок оставьте в предусмотренной ячейке.

Описание решения (не более 1000 символов)
Поезд едет в фоне: вероятностный фильтр по цветовым меткам трассы (старт и полярность неизвестны) ставит его в зону разгрузки, у цели скорость падает до 0,02 м/с. Первый запрос кадра идёт сразу: первый сбой CybP_01 длится около 10 с и проходит, пока подходит корабль. Каждый кадр проверяется по метаданным целостности (status и расхождение сеток expected/observed); повреждённый кадр не используется: пишется предупреждение CybP_01 с frameId, уведомляются dispatcher и crane.1, повтор после cooldown. Контейнеры регистрируются по frame_id первого целого кадра. Тройка по заявке подбирается по сумме, лишние три уходят в буфер, нужные в вагон (блокировка хода означает промах, Y корректируется). Затем поезд с грузом идёт в зону хранения и встаёт по меткам за 12 см до упора, индексы ограничителей защищены; Диспетчер оповещается о готовности в зонах. После доставки отправляется complete_unloading().

In [ ]:
import time, itertools
import numpy as np
import cv2
from quasar_client import QuasarClient, QuasarCommandError, QuasarTimeoutError, QuasarRunStoppedError, QuasarProtocolError
client = QuasarClient.from_environment(timeout=5)
client.wait_for_simulation_start()
client.configure_incidents(['CybP_01'])
client.start_attempt()
T0 = time.monotonic()
def ts(): return round(time.monotonic() - T0, 1)
def log(level, msg, **d):
    print(f'[{ts()}] {level}: {msg}', d if d else '')
    try: client.append_log(level, 'MOD_B', msg[:300], d or None)
    except Exception: pass
def _gen(n=686, w=6):
    s, used, nxt = [], set(), [0]
    while len(s) < n:
        k, c, placed = len(s), nxt[-1], False
        while c < 4:
            if not s or c != s[-1]:
                if k >= w - 1:
                    win = tuple(s[k-w+1:]) + (c,)
                    if win in used: c += 1; continue
                    used.add(win)
                s.append(c); nxt[-1] = c + 1; nxt.append(0); placed = True; break
            c += 1
        if not placed:
            nxt.pop(); last = s.pop(); k = len(s)
            if k >= w - 1: used.discard(tuple(s[k-w+1:]) + (last,))
    return s
SEQ = np.array(_gen()); N = len(SEQ)
G = (-1, 1)
def classify(rgb):
    ch = [rgb['r'], rgb['g'], rgb['b']]; v = max(ch)
    if v < 45: return 0
    if sorted(ch)[1] > 0.6 * v: return None
    return 1 + ch.index(v)
def _shift(a, k):
    out = np.zeros_like(a)
    if k == 0: return a.copy()
    if k > 0: out[k:] = a[:N-k]; out[N-1] += a[N-k:].sum()
    else: out[:N+k] = a[-k:]; out[0] += a[:-k].sum()
    return out
class Loc:
    def __init__(self):
        self.b = np.full((2, N), 0.5 / N); self.t = None; self.v = 0.0
    def predict(self, now):
        if self.t is not None and self.v != 0.0:
            for h in (0, 1):
                d = G[h] * self.v * (now - self.t) / 0.01
                n = int(np.floor(d)); f = d - n
                self.b[h] = (1 - f) * _shift(self.b[h], n) + f * _shift(self.b[h], n + 1)
        for h in (0, 1):
            x = self.b[h]; nb = 0.76 * x
            nb[1:] += 0.12 * x[:-1]; nb[:-1] += 0.12 * x[1:]; nb[0] += 0.12 * x[0]; nb[-1] += 0.12 * x[-1]
            self.b[h] = nb
        self.t = now
    def update(self, obs):
        if obs is None: return
        nb = self.b * np.where(SEQ == obs, 0.8, 0.05)[None, :]; s = nb.sum()
        self.b = nb / s if s > 1e-300 else np.full((2, N), 0.5 / N)
    def est(self):
        m = self.b.sum(axis=0); k = int(np.argmax(m))
        s0, s1 = self.b[0].sum(), self.b[1].sum()
        return k, float(m[max(0, k-2):k+3].sum()), (0 if s0 >= s1 else 1), float(max(s0, s1))
class Train:
    def __init__(self, send, v=0.1, vm=0.05, vf=0.02, far=60, fine=14):
        self.send, self.v, self.vm, self.vf, self.far, self.fine = send, v, vm, vf, far, fine
        self.f = Loc(); self.cmd = 0.0; self.mode = 'idle'; self.est = None; self.target = None
        self.t_mode = 0.0; self.blind = 0.0
    def _set(self, s, now):
        self.f.predict(now); self.f.v = s; self.cmd = s; self.send(s)
    def start(self, target, now):
        self.target = target; self.mode = 'run'; self.f.t = now; self.blind = now; self._set(self.v, now)
    def step(self, rgb, now):
        f = self.f; f.predict(now); f.update(classify(rgb)); k, c, h, ph = f.est(); self.est = k
        if self.mode in ('idle', 'done'): return
        loc = c >= 0.8 and ph >= 0.95
        if self.mode == 'settle':
            if now - self.t_mode >= 0.8: self.mode = 'run' if (loc and k != self.target) else 'done'
            return
        if self.mode == 'rev':
            if now - self.t_mode < 0.3: return
            self.mode = 'run'
        if loc:
            self.blind = now; err = self.target - k
            if self.cmd != 0 and ((G[h] * self.cmd < 0 and k <= LIM_LO) or (G[h] * self.cmd > 0 and k >= LIM_HI)):
                self._set(0.0, now); self.mode = 'settle'; self.t_mode = now; return
            if err == 0:
                self._set(0.0, now); self.mode = 'settle'; self.t_mode = now; return
            need = 1 if err > 0 else -1
            a = abs(err); want = need * G[h] * (self.v if a > self.far else self.vm if a > self.fine else self.vf)
            if want != self.cmd:
                if self.cmd != 0 and (want > 0) != (self.cmd > 0):
                    self._set(0.0, now); self.mode = 'rev'; self.t_mode = now; return
                self._set(want, now)
        else:
            if self.cmd != 0 and now - self.blind > 8: self._set(0.0, now); self.blind = now
            elif self.cmd == 0 and now - self.blind > 1.0: self._set(self.v, now)
def send_speed(s):
    client.execute_gcode('train.1', 'M121' if s == 0 else f'M120 S{s:.2f}', wait=False)
LIM_LO, LIM_HI = 36, 650
STORE = 42
TRAIN_HOME = 336
tr = Train(send_speed)
LAST = {}
def notify(ids, text):
    try: client.notify_nodes(ids, text[:200])
    except Exception as e: log('warning', f'notify: {e}'[:200])
def cyb_status(tag):
    try:
        cy = client.get_state()['cyber']
        inc = [x for x in cy['incidents'] if x['id'] == 'CybP_01']
        print(f'[{ts()}] CYBER {tag}: {inc}')
        print(f'[{ts()}] AUDIT {tag}: {cy.get("auditTrail", [])[-6:]}')
    except Exception as e: log('warning', f'статус инцидентов: {e}'[:200])
def arrived(target, k):
    zone = 'storage zone' if target == STORE else 'unloading zone'
    log('info', f'поезд на месте, готов: {zone}', block=k, t=ts())
    notify(['dispatcher'], f'train.1 ready: {zone}, track block {k}')
def pump():
    client.execute_gcode('train.1', 'M150', wait=False)
    st = client.get_state(); LAST['st'] = st
    if tr.mode == 'done' and LAST.get('tarr') != tr.target: LAST['tarr'] = tr.target; arrived(tr.target, tr.est)
    try: tr.step(st['train']['gcode']['lastResponse']['rgb'], time.monotonic())
    except (KeyError, TypeError): pass
    return st
def nap(sec):
    t = time.monotonic()
    while time.monotonic() - t < sec: pump(); time.sleep(0.03)
MVXY, FXY = 'G1', 9000
MVZ, FZU, FZD = 'G1', 7200, 7200
ZT, ZW = 80, 170
ZC = [None]
BUF_X = 540
BUF_Y = (130, 250, 370, 60, 440)
TIMES = {}
def crane(prog, tmo=40, tag=''):
    client.execute_gcode('crane.1', prog, wait=False)
    t0 = time.monotonic()
    while True:
        st = pump(); g = st['crane']['gcode']
        if g.get('lastError'): raise QuasarCommandError(str(g['lastError']))
        if g.get('busy') is False:
            TIMES.setdefault(tag, []).append(time.monotonic() - t0); return st
        if time.monotonic() - t0 > tmo: raise QuasarTimeoutError('crane timeout')
        time.sleep(0.03)
def clamp(X, Y): return min(max(X, 0), 1106), min(max(Y, 0), 500)
def crane_xy(st): return st['crane']['portal'] * 1000, st['crane']['cart'] * 1000
def mxy(X, Y, Z=None):
    X, Y = clamp(X, Y)
    return f'{MVXY} X{X:.0f} Y{Y:.0f}' + ('' if Z is None else f' Z{Z}') + f' F{FXY}'
def zup(Z): return f'{MVZ} Z{Z} F{FZU}'
def zdn(Z): return f'G1 Z{Z} F{FZD}'
ARUCO = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
CAM = {'last': -10.0, 'hold': 0.0, 'ord': 0}
def cyb_bad(fid, ep, why):
    log('warning', f'ПРЕДУПРЕЖДЕНИЕ CybP_01: нарушена целостность кадра {fid}, кадр отбракован'[:290], t=ts())
    try: client.append_log('warning', 'CybP_01', f'CybP_01 средний уровень, ПРЕДУПРЕЖДЕНИЕ: сбой зрения, кадр {fid} повреждён и отбракован; повтор запроса после cooldown, регистрация только по кадру ok',
                           {'level': 'средний', 'frameId': fid, 'episodeId': ep or '', 'reason': why[:100]})
    except Exception as e: print('append_log', e)
    notify(['dispatcher', 'crane.1'], f'CybP_01: frame {fid} corrupted, rejected, retry after cooldown')
    if not CAM.get('shown'): CAM['shown'] = True; nap(0.5); cyb_status('после первого сбоя')
def cam_try():
    w = max(CAM['last'] + 1.6, CAM['hold']) - time.monotonic()
    if w > 0: nap(w)
    try: o = client.get_crane_camera_observation()
    except QuasarCommandError as e:
        CAM['last'] = time.monotonic()
        if 'cooldown' in str(e): return None
        raise
    except QuasarProtocolError as e:
        CAM['last'] = time.monotonic(); CAM['ord'] += 1
        if 'metadata' not in str(e): raise
        cyb_bad(f"crane-frame-{CAM['ord']}", None, 'invalid-metadata'); return None
    CAM['last'] = time.monotonic(); f = o.fault; ig = f['integrity']; CAM['ord'] = f['captureOrdinal']
    diff = sum(a != b for a, b in zip(ig['expected'], ig['observed']))
    if ig['status'] != 'ok' or diff:
        if f['episodeId'] == 'CybP_01.1' and not CAM['hold']: CAM['hold'] = time.monotonic() + 10.6
        cyb_bad(o.frame_id, f['episodeId'], f"status={ig['status']} diff={diff}"); return None
    return o.frame, o.frame_id
def capture_ok():
    for _ in range(40):
        r = cam_try()
        if r: return r
    raise RuntimeError('камера: нет целого кадра')
IDS = {30, 32, 33, 34, 35, 36}
PX = 1.5
def detect_markers():
    frame, fid = capture_ok()
    cs, ids, _ = cv2.aruco.detectMarkers(frame, ARUCO)
    out = {}
    if ids is not None:
        for cc, i in zip(cs, ids.ravel()):
            if int(i) in IDS: out[int(i)] = (float(cc[0][:, 0].mean()), float(cc[0][:, 1].mean()))
    return out, fid
def scan_at(Y, X=933):
    st = crane(mxy(X, Y, 0), tag='scan')
    Xc, Yc = crane_xy(st)
    mk, fid = detect_markers()
    return {i: (Xc + (320 - cx) / PX, Yc + (240 - cy) / PX, fid) for i, (cx, cy) in mk.items()}
BIAS = [-5.0, 0.0]
def try_grab(X, Y):
    X, Y = clamp(X, Y)
    zs = [100 + 5 * k for k in range(19)] if ZC[0] is None else [ZC[0] + d for d in (0, 5, 10, 20, 35, 50)]
    zs = [z for z in zs if z <= 240]
    try:
        crane([mxy(X, Y, ZT), zdn(zs[0] if ZC[0] is not None else 100)], tag='grab')
        for z in zs:
            try:
                crane([zdn(z), 'M106 P1 S255'], tag='grab')
                if ZC[0] is None: log('info', f'высота захвата найдена: Z={z}')
                ZC[0] = z if ZC[0] is None else min(ZC[0], z)
                return True
            except QuasarCommandError as e:
                if 'blocked' in str(e) or 'limit' in str(e): raise
                crane(['M106 P1 S0'], tag='grab')
        raise QuasarCommandError('magnet-capture-failed на всех высотах')
    except (QuasarCommandError, QuasarTimeoutError) as e:
        st = client.get_state()
        log('warning', f'захват не удался ({X:.0f},{Y:.0f}): {e}', cx=round(st['crane']['portal'] * 1000),
            cy=round(st['crane']['cart'] * 1000), cz=round(st['crane']['zRod'] * 1000))
        try: crane(['M106 P1 S0', zup(ZT)], tag='grab')
        except Exception: pass
        return False
def grab(i, pos):
    X, Y = pos[i]
    if try_grab(X + BIAS[0], Y + BIAS[1]): return True
    for dx, dy in ((-4, 0), (4, 0), (0, 5), (0, -5), (-8, 0), (8, 0)):
        if try_grab(X + BIAS[0] + dx, Y + BIAS[1] + dy):
            BIAS[0] += dx; BIAS[1] += dy; log('info', f'захват {i} со смещением', bx=BIAS[0], by=BIAS[1]); return True
    return False
def to_buffer(k):
    try: crane([zup(ZT), mxy(BUF_X, BUF_Y[k]), zdn((ZC[0] or 130) - 10)], tag='buf')
    except QuasarCommandError as e:
        if 'blocked' not in str(e): raise
    crane(['M106 P1 S0', 'G4 P100', zup(ZT)], tag='buf')
WAGON_X, BAY0_Y, BAY_PITCH = 110, 233, 91
OFFS = (0, 3, -3, 6, -6, 9, -9, 12, -12, 15, -15, 18, -18, 21, -21, 25, -25, 29, -29)
def to_wagon(bay_y):
    for d in OFFS:
        y = bay_y + d
        try:
            crane([zup(ZT), mxy(WAGON_X, y), zdn(ZW)], tag='wag')
        except QuasarCommandError as e:
            if 'blocked' not in str(e): raise
            continue
        crane(['M106 P1 S0', 'G4 P100', zup(ZT)], tag='wag')
        log('info', f'контейнер на вагон: Y={y}', onTrain=client.get_state()['containers']['onTrain'])
        return y
    crane(zup(ZT), tag='wag')
    return None
def agg(items):
    out = {}
    for it in items: out[it['nomenclature']] = out.get(it['nomenclature'], 0) + it['massKg']
    return out
TLIM = 190
TDELIV = 270
completed = False
try:
    log('info', 'старт попытки модуля Б')
    tr.start(TRAIN_HOME, time.monotonic())
    cam_try()
    crane(['G90', mxy(933, 200, 0)], tag='pre')
    while True:
        st = pump()
        if (st.get('moduleA') or {}).get('shipPhase') == 'ready': break
        time.sleep(0.03)
    log('info', 'корабль готов к разгрузке', t=ts())
    pos, FID = {}, {}
    for Y in (200, 420, 120, 480):
        for i, (x, y, fid) in scan_at(Y).items():
            if i not in pos: pos[i] = (x, y); FID[i] = fid
        if len(pos) == 6: break
    log('info', 'маркеры: ' + ' '.join(f'{i}:({p[0]:.0f},{p[1]:.0f})' for i, p in sorted(pos.items())))
    if len(pos) < 6: raise RuntimeError('не найдены все шесть маркеров')
    request = agg(client.get_cargo_request()['items'])
    info = {}
    for i in sorted(pos):
        info[i] = client.get_container_cargo(i)['items']
        resp = client.register_container_cargo(i, info[i], frame_id=FID[i])
        if CAM.get('shown') and i == min(pos):
            try: client.append_log('info', 'CybP_01', f'CybP_01 средний уровень: регистрация по целому кадру {FID[i]} (status ok)', {'level': 'средний', 'frameId': FID[i]})
            except Exception as e: print('append_log', e)
        if resp.get('matchesExpected') is not True: log('error', f'сверка контейнера {i} не прошла: {resp}'[:290])
    need = next((c for c in itertools.combinations(sorted(pos), 3)
                 if agg([it for i in c for it in info[i]]) == request), None)
    if need is None: raise RuntimeError('нет тройки контейнеров под заявку')
    rest = [i for i in sorted(pos) if i not in need]
    log('info', f'зарегистрированы все 6; требуются {list(need)}, в буфер {rest}', t=ts())
    cyb_status('после регистрации')
    done, nbuf, ys = set(), [0], []
    def buffer_pass():
        for i in sorted([j for j in rest if j not in done], key=lambda j: pos[j][1]):
            if time.monotonic() - T0 > TLIM: return
            if grab(i, pos):
                to_buffer(min(nbuf[0], 4)); nbuf[0] += 1; done.add(i)
    def wagon_pass():
        for k, i in enumerate(need):
            if i in done or time.monotonic() - T0 > TLIM: continue
            if k == 0 or not ys: guess = BAY0_Y
            elif k == 1: guess = ys[0] + BAY_PITCH
            else: guess = ys[0] - BAY_PITCH
            if not grab(i, pos): continue
            y = to_wagon(guess)
            if y is None:
                log('error', f'контейнер {i} не встал в карман, сбрасываем в буфер')
                kb = min(nbuf[0], 4); to_buffer(kb); nbuf[0] += 1
                pos[i] = (BUF_X - BIAS[0], BUF_Y[kb] - BIAS[1])
                continue
            done.add(i); ys.append(y)
    buffer_pass()
    while tr.mode != 'done' and time.monotonic() - T0 < TLIM:
        pump(); time.sleep(0.03)
    wagon_pass()
    for _ in range(2):
        if len(done) == 6 or time.monotonic() - T0 > TLIM: break
        log('warning', f'повторный проход, осталось {sorted(set(pos) - done)}')
        buffer_pass(); wagon_pass()
    cs = client.get_state()['containers']
    loaded = cs['onTrain'] == 3 and cs['onShip'] == 0
    crane(zup(0), tag='home')
    if loaded:
        tr.start(STORE, time.monotonic()); log('info', 'груз на составе, поезд идёт в зону хранения', t=ts())
    crane(mxy(0, 0, 0), tag='home')
    while loaded and tr.mode != 'done' and time.monotonic() - T0 < TDELIV:
        pump(); time.sleep(0.03)
    if loaded: log('info' if tr.mode == 'done' else 'error', f'поезд у зоны хранения: блок {tr.est}, цель {STORE}', t=ts())
    cyb_status('перед сигналом')
    cs = client.get_state()['containers']
    if cs['onTrain'] == 3 and cs['onShip'] == 0:
        client.complete_unloading(); completed = True
        log('info', 'разгрузка завершена, сигнал Диспетчеру отправлен', t=ts())
    else:
        log('error', 'разгрузка не завершена, сигнал не отправляем', onTrain=cs['onTrain'], onShip=cs['onShip'], free=cs['free'])
    log('info', 'время по этапам, с/число: ' + ' '.join(f'{k}:{sum(v):.0f}/{len(v)}' for k, v in TIMES.items()))
    t = time.monotonic()
    while time.monotonic() - t < 40:
        if client.get_state()['attemptState'] in ('success', 'failure', 'stopped'): break
        time.sleep(0.3)
    log('info', 'итог попытки: ' + str(client.get_state()['attemptState']), t=ts())
except QuasarRunStoppedError:
    log('info' if completed else 'error', 'прогон остановлен хостом' + (': попытка завершена после сигнала Диспетчеру' if completed else ' до завершения разгрузки'))
except Exception as e:
    log('error', f'сбой сценария: {e!r}')
    raise
finally:
    try: send_speed(0)
    except Exception: pass
